In [ ]:
import run_coarse_design
import parametrization_experiments.parametrization_experiment_helper as parametrization_experiment_helper
import time 
import parallelism, multiprocessing, itertools, setproctitle
import MeshFEM
import os

import inflation, sparse_matrices, mesh, numpy as np, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization
import visualize_stiffness
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib as mpl
import utils, mesh_utilities, benchmark
import shapely
from mesh_utilities import SurfaceSampler, tubeRemesh
import py_newton_optimizer
# from tri_mesh_viewer import TriMeshViewer
from visualization import TriMeshViewerWithSurface
import boundaries
from matplotlib import pyplot as plt
import sheet_optimizer, opt_config
import fabrication
import py_newton_optimizer
import boundaries
import time
import os
import parallelism, multiprocessing, itertools, setproctitle
import serialization_helper
import sys

In [ ]:
num_thread = 8

parallelism.set_max_num_tbb_threads(num_thread)
parallelism.set_gradient_assembly_num_threads(num_thread)
parallelism.set_hessian_assembly_num_threads(num_thread)

In [ ]:
# Update these!
shape_index = 17
pattern_index = 1

# The time stamp should be the one you used in `run_coarse_design.py`
output_time_stamp = 'newton'


In [ ]:

experiment_file, stiffness_path, pattern_name, num_pattern_params, param_index, default_param, param_range, param_normalization_factor, fusing_curve_polyline, shape_name, shape_path, use_holes = parametrization_experiment_helper.parse_input(shape_index, pattern_index)


In [ ]:
output_data_path = 'inverse_design/output/meshing_output_{}/{}_{}/'.format(output_time_stamp, shape_name, pattern_name)


In [ ]:
target_surf = mesh.Mesh(shape_path)
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
target_surf = mesh_utilities.subdivide_loop(target_surf, 1)


In [ ]:
m = MeshFEM.mesh.Mesh(output_data_path + '/parametrized_mesh.obj')
vertices = m.vertices()
m = MeshFEM.mesh.Mesh(np.concatenate((vertices, np.zeros((len(vertices), 1))), axis = 1), m.elements())
fusing_data = np.load(output_data_path + '/fusing_data.npy')


In [ ]:
isheet = inflation.InflatableSheet(m, fusing_data)
uv = np.load(output_data_path + '/rparam_uv.npy')

paramSampler = SurfaceSampler(np.pad(uv, [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())
isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

sys.path.append("../../")

# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

bdryVars = boundaries.getOuterBoundaryVars(isheet)
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)

In [ ]:
# First generate results with fixed boundary 
viewer = TriMeshViewerWithSurface(isheet, target_surf, width=768, height=640)
viewer.showWireframe(True)

# viewer.setCameraParams(((1.613494603240345, -3.9332708615926393, 1.4922998234349831),
# (-0.05948468564942635, 0.33267929672385665, 0.941162078339598),
# (0.0, 0.0, 0.0)))

viewer.update(scalarField=utils.getStrains(isheet)[:, 0])   
viewer.show()

In [ ]:

print("Fix feet!")
m = isheet.mesh()
V = m.vertices()
BV = m.boundaryVertices()
arclen = lambda l: np.linalg.norm(np.diff(V[BV[np.array(l)]], axis=0), axis=1).sum()
outerLoopBdryVertices = max(m.boundaryLoops(), key=arclen)
feet_fixed_vars = []
for bvi in outerLoopBdryVertices:
    for c in range(3):
        feet_fixed_vars.append(isheet.varIdx(0, BV[bvi], c))

feet_fixed_vars = np.array(feet_fixed_vars).reshape((-1, 3))

sheet_vars = isheet.getVars()

fixed_vars_values = sheet_vars[feet_fixed_vars[:, 1]]

bottom_fixed_vars = []
for i in range(len(feet_fixed_vars)):
    if np.abs(fixed_vars_values[i] - min(fixed_vars_values))< 1:
        bottom_fixed_vars.extend(feet_fixed_vars[i])
bdryVars = np.array(bottom_fixed_vars)

fixedVars_list = [bdryVars, []]

In [ ]:
framerate = 10
def cb(it):
    if it % framerate == 0:
        viewer.update()


# ### First solve with low pressure to get out of indefinite state
benchmark.reset()

isheet.pressure = 1e-2
opts.niter = 20
cr = inflation.inflation_newton(isheet, bdryVars, opts, hessianShift = 0, callback = cb)
benchmark.report()

fixedVars_list = [bdryVars, []]
tag_name = ['fixed_boundary', 'free_boundary']
hessian_shifts = [0, 1e-6]

In [ ]:
# for i in range(len(fixedVars_list)):
for i in range(1):
    fixedVars = fixedVars_list[i]
    tag = tag_name[i]
    hessian_shift = hessian_shifts[i]
#     if os.path.exists(output_data_path + '/{}_inflated_sheet_vars.npy'.format(tag)):
#         isheet.setVars(np.load(output_data_path + '/{}_inflated_sheet_vars.npy'.format(tag)))
#         viewer.update()  

#     else:
    # ### Then inflate
    isheet.pressure = 0.025
    opts.niter = 200
    opts.gradTol = 1e-7

    benchmark.reset()
    cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessian_shift, callback = cb)
    benchmark.report()

    np.save(output_data_path + '/{}_inflated_sheet_vars.npy'.format(tag), isheet.getVars())

orender = viewer.offscreenRenderer(width=1024,height=1024)
orender.render()
orender.save(output_data_path + '/{}_parametrized_mesh_inflated.png'.format(tag))

def export_top_bottom_mesh(isheet, export_path, shape_name, pattern_name):
    mesh_3d = isheet.visualizationMesh(True)
    mesh_2d = isheet.mesh()
    vx_3d = mesh_3d.vertices()
    elements_3d = mesh_3d.elements()

    new_mesh_3d = MeshFEM.Mesh(vx_3d[:mesh_2d.numVertices()], elements_3d[:mesh_2d.numElements()])

    new_mesh_3d.save(export_path + '/{}_{}_{}_mesh_3d_top.obj'.format(tag, shape_name, pattern_name))

    new_mesh_3d = MeshFEM.Mesh(vx_3d[m.numVertices():], elements_3d[mesh_2d.numElements():] - mesh_2d.numVertices())
    new_mesh_3d.save(export_path + '/{}_{}_{}_mesh_3d_bottom.obj'.format(tag, shape_name, pattern_name))

export_top_bottom_mesh(isheet, output_data_path, shape_name, pattern_name)

In [ ]:
benchmark.report()

In [ ]:
cr.success